In [1]:
from pathlib import Path
import pandas as pd

In [2]:

# 1. BASE DIRECTORY 

BASE = Path.home() / "Desktop" / "LCS_eligibility_Ireland"

CODE_DIR = BASE / "code"
RAW_DIR = BASE / "data_raw"
PROC_DIR = BASE / "data_processed"
OUT_DIR = BASE / "output"

print("Using base directory:", BASE)


Using base directory: C:\Users\tatianabezdenezhnykh\Desktop\LCS_eligibility_Ireland


In [3]:

# Load the newly provided dataset for review
file_path = RAW_DIR / "projections2057_raw.csv"
proj_df  = pd.read_csv(file_path)

# Show first few rows to understand structure
proj_df .head()

#5,183,966 people

#5,149,139 cesus

,Statistic Label,Year,Age,Sex,Criteria for Projection,UNIT,VALUE
0,Population Projections based on Census 2022,2022,Under 1 year,Male,Method - M2,Number,29546
1,Population Projections based on Census 2022,2022,Under 1 year,Female,Method - M2,Number,28138
2,Population Projections based on Census 2022,2022,1 year,Male,Method - M2,Number,28985
3,Population Projections based on Census 2022,2022,1 year,Female,Method - M2,Number,27646
4,Population Projections based on Census 2022,2022,2 years,Male,Method - M2,Number,30263


In [4]:
# Dropping unnecessary columns from the population projection dataset
columns_to_drop = ['Statistic Label', 'Criteria for Projection', 'UNIT']
proj_df_cleaned = proj_df.drop(columns=columns_to_drop)

In [5]:
# Renaming columns for better clarity and consistency
proj_df_cleaned = proj_df_cleaned.rename(columns={
    'Year': 'year',
    'Age': 'age',
    'Sex': 'gender',
    'VALUE': 'population'
})

# Displaying unique values of categorical variables
categorical_vars = ['year', 'age', 'gender']
unique_values = {var: proj_df_cleaned[var].unique() for var in categorical_vars}
unique_values


{'year': array([2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032,
        2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043,
        2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054,
        2055, 2056, 2057], dtype=int64),
 'age': array(['Under 1 year', '1 year', '2 years', '3 years', '4 years',
        '5 years', '6 years', '7 years', '8 years', '9 years', '10 years',
        '11 years', '12 years', '13 years', '14 years', '15 years',
        '16 years', '17 years', '18 years', '19 years', '20 years',
        '21 years', '22 years', '23 years', '24 years', '25 years',
        '26 years', '27 years', '28 years', '29 years', '30 years',
        '31 years', '32 years', '33 years', '34 years', '35 years',
        '36 years', '37 years', '38 years', '39 years', '40 years',
        '41 years', '42 years', '43 years', '44 years', '45 years',
        '46 years', '47 years', '48 years', '49 years', '50 years',
        '51 years', '52 years', '53 

In [6]:


# Creating proper 5-year age bands
def age_band(age):
    if age in ['Under 1 year', '0 years', '1 year', '2 years', '3 years', '4 years']:
        return '0-4'
    elif age == '100 years and over':
        return '100+'
    elif 'year' in age:
        try:
            num = int(age.split()[0])
            lower = 5 * (num // 5)
            upper = lower + 4
            return f'{lower}-{upper}'
        except:
            return None
    else:
        return None


# Applying age banding
proj_df_cleaned['age_group'] = proj_df_cleaned['age'].apply(age_band)

# Aggregate population by year, age group, and gender
agg_proj = proj_df_cleaned.groupby(['year', 'age_group', 'gender'], as_index=False)['population'].sum()

# Sort for neatness
agg_proj = agg_proj.sort_values(by=['year', 'gender', 'age_group'])

agg_proj.head()

agg_proj.to_csv(PROC_DIR / "projections2057.csv", index=False)



In [7]:
agg_proj.head(1000)

,year,age_group,gender,population
0,2022,0-4,Female,144247
2,2022,10-14,Female,183366
4,2022,15-19,Female,165149
6,2022,20-24,Female,153935
8,2022,25-29,Female,150042
...,...,...,...,...
991,2046,75-79,Male,141358
993,2046,80-84,Male,106423
995,2046,85-89,Male,66997
997,2046,90-94,Male,33027
